In [1]:
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import sys
import time
import os
!pip install ipython-autotime
%load_ext autotime
!pip install torchinfo
from torchinfo import summary

time: 869 ms (started: 2026-04-21 02:17:24 +07:00)


In [2]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.11.0+cu128
True
NVIDIA GeForce RTX 5090
time: 124 ms (started: 2026-04-21 02:17:29 +07:00)


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# 1.  PointNet++-style geometry encoder
# ─────────────────────────────────────────────────────────────────────────────

def square_distance(src: torch.Tensor, dst: torch.Tensor) -> torch.Tensor:
    """[B,N,3] × [B,M,3] → [B,N,M] squared L2 distances."""
    dist = -2 * torch.bmm(src, dst.transpose(1, 2))
    dist += src.pow(2).sum(-1, keepdim=True)
    dist += dst.pow(2).sum(-1, keepdim=True).transpose(1, 2)
    return dist.clamp(min=0)


def index_points(points: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
    """Gather rows from points using idx (arbitrary shape suffix)."""
    B = points.shape[0]
    view = [B] + [1] * (idx.dim() - 1)
    batch = torch.arange(B, device=points.device).view(view).expand_as(idx)
    return points[batch, idx]


def fps(xyz: torch.Tensor, n: int) -> torch.Tensor:
    """Farthest-point sampling; xyz [B,N,3] → indices [B,n]."""
    B, N, _ = xyz.shape
    ids = torch.zeros(B, n, dtype=torch.long, device=xyz.device)
    dist = xyz.new_full((B, N), 1e10)
    cur = torch.randint(0, N, (B,), device=xyz.device)
    for i in range(n):
        ids[:, i] = cur
        c = xyz[torch.arange(B), cur].unsqueeze(1)      # [B,1,3]
        d = (xyz - c).pow(2).sum(-1)                    # [B,N]
        dist = torch.minimum(dist, d)
        cur = dist.argmax(dim=1)
    return ids


def knn(k: int, xyz: torch.Tensor, query: torch.Tensor) -> torch.Tensor:
    """query [B,S,3], xyz [B,N,3] → [B,S,k] neighbour indices."""
    d = square_distance(query, xyz)                     # [B,S,N]
    return d.topk(k, dim=-1, largest=False).indices


class SetAbstraction(nn.Module):
    """
    Single PointNet++ Set Abstraction block.
    Supports 3-D XYZ input + optional feature channels.
    """
    def __init__(self, n_sample: int, k: int, in_ch: int, mlp_dims: Tuple[int, ...]):
        super().__init__()
        self.n_sample = n_sample
        self.k = k
        layers: list[nn.Module] = []
        c_prev = in_ch
        for c in mlp_dims:
            layers += [nn.Conv2d(c_prev, c, 1), nn.BatchNorm2d(c), nn.GELU()]
            c_prev = c
        self.mlp = nn.Sequential(*layers)
        self.out_ch = mlp_dims[-1]

    def forward(
        self, xyz: torch.Tensor, feat: Optional[torch.Tensor]
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        xyz  [B,N,3], feat [B,N,Cin] or None
        → new_xyz [B,S,3], new_feat [B,S,Cout]
        """
        S = self.n_sample
        ids = fps(xyz, S)                       # [B,S]
        new_xyz = index_points(xyz, ids)        # [B,S,3]

        nn_ids = knn(self.k, xyz, new_xyz)      # [B,S,k]
        nb_xyz = index_points(xyz, nn_ids)      # [B,S,k,3]
        nb_xyz_rel = nb_xyz - new_xyz.unsqueeze(2)

        if feat is None:
            x = nb_xyz_rel                      # [B,S,k,3]
        else:
            nb_feat = index_points(feat, nn_ids)
            x = torch.cat([nb_xyz_rel, nb_feat], dim=-1)

        # Conv over local points
        x = x.permute(0, 3, 1, 2)              # [B,C,S,k]
        x = self.mlp(x)                         # [B,Cout,S,k]
        x = x.max(dim=-1).values.permute(0, 2, 1)  # [B,S,Cout]
        return new_xyz, x


class GeometryEncoder(nn.Module):
    """
    Hierarchical PointNet++ geometry encoder.
    Input: xyz [B,N,3], feat [B,N,3] (normals).
    Output: geo_tokens [B,S2,D]
    """
    def __init__(self, d_model: int = 256):
        super().__init__()
        # Two SA levels:  4096 → 512 → 128  (defaults; caller may override)
        # First SA: xyz+normals → 64-dim tokens
        self.sa1 = SetAbstraction(
            n_sample=512, k=32,
            in_ch=6,            # 3 relative xyz + 3 normals
            mlp_dims=(64, 64, 128),
        )
        # Second SA: 512-token → 128-token, 128→256
        self.sa2 = SetAbstraction(
            n_sample=128, k=32,
            in_ch=128 + 3,      # 3 relative xyz + 128 features from SA1
            mlp_dims=(128, 256, d_model),
        )
        self.d_model = d_model

    def forward(self, xyz: torch.Tensor, normals: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        feat = normals                              # [B,N,3]
        xyz1, feat1 = self.sa1(xyz, feat)          # [B,512,128]
        xyz2, feat2 = self.sa2(xyz1, feat1)        # [B,128,D]
        return xyz2, feat2                         # positions & tokens


# ─────────────────────────────────────────────────────────────────────────────
# 2.  Disk-image encoder  (shared CNN → positional embed → Transformer)
# ─────────────────────────────────────────────────────────────────────────────

class PatchEmbedCNN(nn.Module):
    """
    Lightweight CNN that maps an n_pix×n_pix disk image to a 1-D token.
    Uses strided convolutions instead of literal patch splitting so it
    remains differentiable and handles circular (disk-shaped) inputs gracefully.

    Input:  [B*N, C_in, n_pix, n_pix]
    Output: [B*N, d_model]
    """
    def __init__(self, c_in: int = 3, n_pix: int = 64, d_model: int = 256):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(c_in, 32,  3, stride=2, padding=1),   # n_pix/2
            nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32,  64,  3, stride=2, padding=1),    # n_pix/4
            nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64,  128, 3, stride=2, padding=1),    # n_pix/8
            nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128, d_model, 3, stride=2, padding=1),# n_pix/16
            nn.BatchNorm2d(d_model), nn.GELU(),
            nn.AdaptiveAvgPool2d(1),                         # global pool
        )
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B*N, C, H, W]
        z = self.stem(x).flatten(1)     # [B*N, d_model]
        return self.proj(z)             # [B*N, d_model]


class ImageEncoder(nn.Module):
    """
    Per-point disk-image encoder.

    Flow:
        1. Reshape N per-point images into a batch dimension.
        2. CNN patch-embed → 1 token per point.
        3. Optionally add a learnable positional embedding (tied to xy-grid).
        4. Transformer self-attention over all N image tokens.

    Input:  images [B, N, C, H, W]
    Output: img_tokens [B, N', D]  where N' = N (all tokens kept)

    NOTE: For memory efficiency the default only encodes a *sampled subset*
    of S_img points (same FPS indices as the geometry branch), not all N.
    The caller should pre-select images before passing here.
    """
    def __init__(
        self,
        c_in: int = 3,
        n_pix: int = 64,
        d_model: int = 256,
        n_img_tokens: int = 128,    # matches geometry branch S2=128
        n_heads: int = 8,
        n_layers: int = 4,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.n_img_tokens = n_img_tokens
        self.patch_embed = PatchEmbedCNN(c_in, n_pix, d_model)

        # Learnable positional embedding over N_img_tokens positions
        self.pos_embed = nn.Parameter(torch.randn(1, n_img_tokens, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True,
            activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """
        images: [B, S, C, H, W]   S = n_img_tokens sampled points
        returns: [B, S, D]
        """
        B, S, C, H, W = images.shape
        x = images.view(B * S, C, H, W)
        tokens = self.patch_embed(x).view(B, S, -1)    # [B,S,D]
        tokens = tokens + self.pos_embed[:, :S]
        tokens = self.transformer(tokens)
        return self.norm(tokens)                        # [B,S,D]


# ─────────────────────────────────────────────────────────────────────────────
# 3.  Cross-attention fusion
# ─────────────────────────────────────────────────────────────────────────────

class CrossAttentionFusion(nn.Module):
    """
    Bidirectional cross-attention between geometry and image token sequences.
    geo_tokens query image_tokens → refined_geo
    img_tokens query geo_tokens  → refined_img
    Both get a residual FFN afterwards.
    """
    def __init__(self, d_model: int, n_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        self.attn_g2i = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.attn_i2g = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)

        self.ffn_g = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
        )
        self.ffn_i = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
        )
        self.norm_g1 = nn.LayerNorm(d_model)
        self.norm_g2 = nn.LayerNorm(d_model)
        self.norm_i1 = nn.LayerNorm(d_model)
        self.norm_i2 = nn.LayerNorm(d_model)

    def forward(
        self, geo: torch.Tensor, img: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # geo queries img
        a_g, _ = self.attn_g2i(geo, img, img, need_weights=False)
        geo = self.norm_g1(geo + a_g)
        geo = self.norm_g2(geo + self.ffn_g(geo))

        # img queries geo
        a_i, _ = self.attn_i2g(img, geo, geo, need_weights=False)
        img = self.norm_i1(img + a_i)
        img = self.norm_i2(img + self.ffn_i(img))

        return geo, img


# ─────────────────────────────────────────────────────────────────────────────
# 4.  Full multimodal classifier
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class MultimodalNetCfg:
    d_model:        int   = 256
    n_geo_tokens:   int   = 128     # SA2 output count
    n_img_tokens:   int   = 128     # must match n_geo_tokens for simple fusion
    geo_heads:      int   = 8
    geo_layers:     int   = 4
    img_heads:      int   = 8
    img_layers:     int   = 4
    fusion_heads:   int   = 8
    n_fusion_layers:int   = 2
    n_pix:          int   = 64      # disk image resolution
    img_channels:   int   = 3       # curvature channels (e.g. k1,k2,shape_idx)
    n_classes:      int   = 40
    dropout:        float = 0.1
    cls_hidden:     int   = 512


class GeometryTransformer(nn.Module):
    """Self-attention transformer on geometry tokens."""
    def __init__(self, d_model: int, n_heads: int, n_layers: int, dropout: float):
        super().__init__()
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.norm(self.enc(x))


class MultimodalNet(nn.Module):
    """
    Multimodal point cloud + disk image classifier for ModelNet40.

    Forward signature
    -----------------
    xyz    : [B, N, 3]
    normals: [B, N, 3]
    images : [B, S, C, H, W]   pre-selected S=n_img_tokens disk images per shape
                                  (caller selects the FPS-subsampled subset; see
                                   ModelNet40Collator below)

    Returns
    -------
    logits : [B, n_classes]
    """

    def __init__(self, cfg: Optional[MultimodalNetCfg] = None):
        super().__init__()
        if cfg is None:
            cfg = MultimodalNetCfg()
        self.cfg = cfg
        D = cfg.d_model

        # ── geometry branch ────────────────────────────────────────────────
        self.geo_encoder  = GeometryEncoder(d_model=D)
        self.geo_transformer = GeometryTransformer(D, cfg.geo_heads, cfg.geo_layers, cfg.dropout)

        # ── image branch ──────────────────────────────────────────────────
        self.img_encoder = ImageEncoder(
            c_in=cfg.img_channels,
            n_pix=cfg.n_pix,
            d_model=D,
            n_img_tokens=cfg.n_img_tokens,
            n_heads=cfg.img_heads,
            n_layers=cfg.img_layers,
            dropout=cfg.dropout,
        )

        # ── cross-attention fusion ────────────────────────────────────────
        self.fusion_layers = nn.ModuleList([
            CrossAttentionFusion(D, cfg.fusion_heads, cfg.dropout)
            for _ in range(cfg.n_fusion_layers)
        ])

        # ── classification head ───────────────────────────────────────────
        # max-pool + avg-pool from both branches → 4D concat
        in_dim = D * 4
        self.cls_head = nn.Sequential(
            nn.Linear(in_dim, cfg.cls_hidden),
            nn.BatchNorm1d(cfg.cls_hidden),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.cls_hidden, cfg.cls_hidden // 2),
            nn.BatchNorm1d(cfg.cls_hidden // 2),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.cls_hidden // 2, cfg.n_classes),
        )

    def _pool(self, tokens: torch.Tensor) -> torch.Tensor:
        """max-pool + avg-pool → [B, 2D]."""
        return torch.cat([tokens.max(1).values, tokens.mean(1)], dim=1)

    def forward(
        self,
        xyz:     torch.Tensor,   # [B, N, 3]
        normals: torch.Tensor,   # [B, N, 3]
        images:  torch.Tensor,   # [B, S, C, H, W]
    ) -> torch.Tensor:

        # ── geometry branch ────────────────────────────────────────────────
        _, geo_tokens = self.geo_encoder(xyz, normals)          # [B, S, D]
        geo_tokens = self.geo_transformer(geo_tokens)           # [B, S, D]

        # ── image branch ──────────────────────────────────────────────────
        img_tokens = self.img_encoder(images)                   # [B, S, D]

        # ── bidirectional cross-attention ─────────────────────────────────
        for layer in self.fusion_layers:
            geo_tokens, img_tokens = layer(geo_tokens, img_tokens)

        # ── aggregate & classify ──────────────────────────────────────────
        feat = torch.cat([self._pool(geo_tokens), self._pool(img_tokens)], dim=1)  # [B, 4D]
        return self.cls_head(feat)                              # [B, n_classes]

    @torch.no_grad()
    def predict(self, xyz, normals, images) -> Tuple[torch.Tensor, torch.Tensor]:
        """Returns (class_idx, confidence) tensors."""
        logits = self.forward(xyz, normals, images)
        probs = F.softmax(logits, dim=-1)
        conf, idx = probs.max(dim=-1)
        return idx, conf


# ─────────────────────────────────────────────────────────────────────────────
# 5.  Data collator — bridges ModelNet40Dataset → MultimodalNet
# ─────────────────────────────────────────────────────────────────────────────

class ModelNet40Collator:
    """
    Collate function that:
      1. Stacks standard ModelNet40Dataset batch keys.
      2. Loads per-point disk images from disk (path pattern: {name}_{pid}.png).
      3. Subsamples S_img points (by FPS on xyz) and returns their images.

    Usage
    -----
        collator = ModelNet40Collator(img_dir="./cache_images", S=128, n_pix=64)
        loader = DataLoader(ds, batch_size=8, collate_fn=collator)
        for batch in loader:
            logits = model(batch["xyz"], batch["normals"], batch["images"])
    """

    def __init__(
        self,
        img_dir: str,
        S: int  = 128,      # how many images to use per shape
        n_pix: int = 64,
        n_channels: int = 3,
        img_ext: str = ".png",
    ):
        self.img_dir   = img_dir
        self.S         = S
        self.n_pix     = n_pix
        self.n_channels = n_channels
        self.ext       = img_ext

    def _load_image(self, name: str, pid: int) -> torch.Tensor:
        """Load a single disk image → [C, H, W] float32 in [0,1]."""
        import os
        from PIL import Image
        import numpy as np
        path = os.path.join(self.img_dir, f"{name}_{pid}{self.ext}")
        if not os.path.exists(path):
            # Return a blank image if missing (e.g. near-boundary points)
            return torch.zeros(self.n_channels, self.n_pix, self.n_pix)
        img = Image.open(path).convert("RGB" if self.n_channels == 3 else "L")
        img = img.resize((self.n_pix, self.n_pix))
        arr = np.array(img, dtype=np.float32) / 255.0
        if arr.ndim == 2:
            arr = arr[None]                 # [1,H,W]
        else:
            arr = arr.transpose(2, 0, 1)   # [C,H,W]
        return torch.from_numpy(arr[:self.n_channels])

    @staticmethod
    def _fps_indices(xyz: torch.Tensor, S: int) -> torch.Tensor:
        """Simple FPS on a single [N,3] tensor → [S] indices."""
        N = xyz.shape[0]
        if S >= N:
            return torch.arange(N)
        ids  = torch.zeros(S, dtype=torch.long)
        dist = torch.full((N,), 1e10)
        cur  = 0
        for i in range(S):
            ids[i] = cur
            c = xyz[cur]
            d = (xyz - c).pow(2).sum(-1)
            dist = torch.minimum(dist, d)
            cur  = int(dist.argmax())
        return ids

    def __call__(self, samples: list[dict]) -> dict:
        batch_xyz, batch_norms, batch_imgs, batch_labels = [], [], [], []

        for s in samples:
            pts   = s["points"]    # [N,3]
            norms = s["normals"]   # [N,3]
            label = s["label"]
            name  = s["name"]
            N = pts.shape[0]

            # FPS to select S representative points
            ids = self._fps_indices(pts, self.S)    # [S]

            batch_xyz.append(pts.unsqueeze(0))       # will stack [N,3]
            batch_norms.append(norms.unsqueeze(0))

            # Load images for selected points
            imgs = torch.stack([
                self._load_image(name, int(ids[i]))
                for i in range(len(ids))
            ])  # [S, C, H, W]
            batch_imgs.append(imgs.unsqueeze(0))     # [1,S,C,H,W]
            batch_labels.append(label if isinstance(label, torch.Tensor) else torch.tensor(label))

        return {
            "xyz":     torch.cat(batch_xyz,   dim=0),   # [B,N,3]
            "normals": torch.cat(batch_norms, dim=0),   # [B,N,3]
            "images":  torch.cat(batch_imgs,  dim=0),   # [B,S,C,H,W]
            "label":   torch.stack(batch_labels),        # [B]
        }


# ─────────────────────────────────────────────────────────────────────────────
# 6.  Training utilities
# ─────────────────────────────────────────────────────────────────────────────

def build_optimizer(model: MultimodalNet, lr: float = 1e-3, weight_decay: float = 1e-4):
    """AdamW with separate LR for backbone vs head."""
    head_params  = list(model.cls_head.parameters())
    head_ids     = {id(p) for p in head_params}
    body_params  = [p for p in model.parameters() if id(p) not in head_ids]

    return torch.optim.AdamW([
        {"params": body_params, "lr": lr,       "weight_decay": weight_decay},
        {"params": head_params, "lr": lr * 10,  "weight_decay": weight_decay},
    ])


def train_one_epoch(
    model:     MultimodalNet,
    loader:    DataLoader,
    opt:       torch.optim.Optimizer,
    device:    str,
    label_smoothing: float = 0.1,
) -> dict:
    model.train()
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    total_loss = total_correct = total_n = 0

    for batch in loader:
        xyz    = batch["xyz"].to(device)
        norms  = batch["normals"].to(device)
        images = batch["images"].to(device)
        labels = batch["label"].to(device)

        opt.zero_grad(set_to_none=True)
        logits = model(xyz, norms, images)
        loss   = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        B = xyz.shape[0]
        total_loss    += loss.item() * B
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_n       += B

    return {
        "loss": total_loss / total_n,
        "acc":  total_correct / total_n,
    }


@torch.no_grad()
def evaluate(model: MultimodalNet, loader: DataLoader, device: str) -> dict:
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = total_correct = total_n = 0

    for batch in loader:
        xyz    = batch["xyz"].to(device)
        norms  = batch["normals"].to(device)
        images = batch["images"].to(device)
        labels = batch["label"].to(device)

        logits = model(xyz, norms, images)
        loss   = criterion(logits, labels)

        B = xyz.shape[0]
        total_loss    += loss.item() * B
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_n       += B

    return {
        "loss": total_loss / total_n,
        "acc":  total_correct / total_n,
    }



time: 5.81 ms (started: 2026-04-21 02:17:34 +07:00)


In [4]:
MODELNET_ROOT = "out/ModelNet40"
CSV_PATH      = "modelnet10_better_subset.csv"
IMG_DIR       = "/train_modelnet10_local_subset"          # output of _export_to_image()
CKPT_DIR      = "./checkpoints_local"
os.makedirs(CKPT_DIR, exist_ok=True)

time: 804 μs (started: 2026-04-21 02:17:54 +07:00)


In [5]:
from modelnet40_dataset import ModelNet40Dataset
train_ds = ModelNet40Dataset(
    root=MODELNET_ROOT, csv_path=CSV_PATH, split="train",
    n_points=1024, normalize=True, augment=False, compute_feats=True,
)
val_ds = ModelNet40Dataset(
    root=MODELNET_ROOT, csv_path=CSV_PATH, split="test",
    n_points=1024, normalize=True, augment=False, compute_feats=True,
)

time: 245 ms (started: 2026-04-21 02:17:57 +07:00)


In [6]:
train_ds.preprocess_all(cache_dir="cache_modelnet10_subset")
val_ds.preprocess_all(cache_dir="cache_modelnet10_subset")

[ModelNet40Dataset] Preprocessing 500 'train' samples → cache_modelnet10_subset


preprocess(train): 100%|████████████████████████████████████████████████████████████| 500/500 [00:00<00:00, 5523.01it/s]


[ModelNet40Dataset] Done. 500 samples ready.
[ModelNet40Dataset] Preprocessing 100 'test' samples → cache_modelnet10_subset


preprocess(test): 100%|█████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 5444.96it/s]

[ModelNet40Dataset] Done. 100 samples ready.
time: 115 ms (started: 2026-04-21 02:17:57 +07:00)


In [11]:
collator = ModelNet40Collator(img_dir=IMG_DIR, S=128, n_pix=64*2, n_channels=3)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,
                          num_workers=2, collate_fn=collator, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False,
                          num_workers=2, collate_fn=collator)

time: 586 μs (started: 2026-04-21 02:18:52 +07:00)


In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda
time: 256 μs (started: 2026-04-21 02:18:53 +07:00)


In [13]:
cfg = MultimodalNetCfg(
    d_model=256, n_geo_tokens=128, n_img_tokens=128,
    geo_heads=8, geo_layers=4, img_heads=8, img_layers=4,
    fusion_heads=8,n_fusion_layers=2,
    n_pix=64*2, img_channels=3, n_classes=10,
    dropout=0.1, cls_hidden=512,
)
model = MultimodalNet(cfg).to(device)

time: 79.9 ms (started: 2026-04-21 02:18:54 +07:00)


In [10]:
batch = next(iter(train_loader))
for k, v in batch.items():
    print(k, v.shape)
input_data = {k: v.to(device) for k, v in batch.items() if k!='label'}
summary(model,input_data=input_data)

xyz torch.Size([16, 1024, 3])
normals torch.Size([16, 1024, 3])
images torch.Size([16, 128, 3, 128, 128])
label torch.Size([16])


Layer (type:depth-idx)                             Output Shape              Param #
MultimodalNet                                      [16, 10]                  --
├─GeometryEncoder: 1-1                             [16, 128, 3]              --
│    └─SetAbstraction: 2-1                         [16, 512, 3]              --
│    │    └─Sequential: 3-1                        [16, 128, 512, 32]        13,440
│    └─SetAbstraction: 2-2                         [16, 128, 3]              --
│    │    └─Sequential: 3-2                        [16, 256, 128, 32]        116,992
├─GeometryTransformer: 1-2                         [16, 128, 256]            --
│    └─TransformerEncoder: 2-3                     [16, 128, 256]            --
│    │    └─ModuleList: 3-3                        --                        3,159,040
│    └─LayerNorm: 2-4                              [16, 128, 256]            512
├─ImageEncoder: 1-3                                [16, 128, 256]            32,768
│    └─PatchEm

time: 1.76 s (started: 2026-04-21 02:18:34 +07:00)


In [14]:
opt  = build_optimizer(model, lr=1e-3)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=200)
best_acc = 0.0

time: 405 ms (started: 2026-04-21 02:18:57 +07:00)


In [15]:
for epoch in range(1, 101):
    tr = train_one_epoch(model, train_loader, opt, device)
    va = evaluate(model, val_loader, device)
    sched.step()
    print(f"[{epoch:03d}] train loss={tr['loss']:.4f} acc={tr['acc']:.4f} | "
          f"val loss={va['loss']:.4f} acc={va['acc']:.4f}")
    if va["acc"] > best_acc:
        best_acc = va["acc"]
        torch.save({"epoch": epoch, "model": model.state_dict(),
                    "val_acc": best_acc}, f"{CKPT_DIR}/best.pt")
        print(f"  ↑ saved  val_acc={best_acc:.4f}")

[001] train loss=2.1359 acc=0.3105 | val loss=23.0809 acc=0.1000
  ↑ saved  val_acc=0.1000
[002] train loss=1.8452 acc=0.4234 | val loss=16.6958 acc=0.1000
[003] train loss=1.6728 acc=0.4819 | val loss=3.4622 acc=0.1600
  ↑ saved  val_acc=0.1600
[004] train loss=1.7004 acc=0.4556 | val loss=15.5647 acc=0.1000
[005] train loss=1.7038 acc=0.4153 | val loss=8.0066 acc=0.2300
  ↑ saved  val_acc=0.2300
[006] train loss=1.5576 acc=0.5101 | val loss=8.2207 acc=0.2000
[007] train loss=1.5893 acc=0.4899 | val loss=4.1664 acc=0.1000
[008] train loss=1.4773 acc=0.5585 | val loss=4.2386 acc=0.1900
[009] train loss=1.4453 acc=0.5685 | val loss=2.2264 acc=0.2700
  ↑ saved  val_acc=0.2700
[010] train loss=1.4736 acc=0.5504 | val loss=3.5448 acc=0.2400
[011] train loss=1.4196 acc=0.5766 | val loss=7.9126 acc=0.1000
[012] train loss=1.3508 acc=0.6089 | val loss=3.1012 acc=0.2700
[013] train loss=1.3543 acc=0.6149 | val loss=2.9840 acc=0.3100
  ↑ saved  val_acc=0.3100
[014] train loss=1.2591 acc=0.6331 

In [16]:
for epoch in range(101, 151):
    tr = train_one_epoch(model, train_loader, opt, device)
    va = evaluate(model, val_loader, device)
    sched.step()
    print(f"[{epoch:03d}] train loss={tr['loss']:.4f} acc={tr['acc']:.4f} | "
          f"val loss={va['loss']:.4f} acc={va['acc']:.4f}")
    if va["acc"] > best_acc:
        best_acc = va["acc"]
        torch.save({"epoch": epoch, "model": model.state_dict(),
                    "val_acc": best_acc}, f"{CKPT_DIR}/best_more100.pt")
        print(f"  ↑ saved  val_acc={best_acc:.4f}")

[101] train loss=0.6415 acc=0.9556 | val loss=0.8056 acc=0.7900
[102] train loss=0.6833 acc=0.9395 | val loss=0.9533 acc=0.7500
[103] train loss=0.6110 acc=0.9637 | val loss=0.8335 acc=0.7800
[104] train loss=0.6085 acc=0.9758 | val loss=0.8004 acc=0.8300
  ↑ saved  val_acc=0.8300
[105] train loss=0.6140 acc=0.9657 | val loss=0.8784 acc=0.8000
[106] train loss=0.5933 acc=0.9758 | val loss=0.9031 acc=0.7800
[107] train loss=0.6390 acc=0.9597 | val loss=0.7662 acc=0.8100
[108] train loss=0.6020 acc=0.9738 | val loss=0.6662 acc=0.8200
[109] train loss=0.5904 acc=0.9758 | val loss=0.7918 acc=0.7900
[110] train loss=0.5903 acc=0.9819 | val loss=0.8321 acc=0.7600
[111] train loss=0.5682 acc=0.9859 | val loss=0.8127 acc=0.8200
[112] train loss=0.5856 acc=0.9819 | val loss=0.8440 acc=0.8000
[113] train loss=0.6139 acc=0.9677 | val loss=0.8969 acc=0.7800
[114] train loss=0.5829 acc=0.9859 | val loss=0.7046 acc=0.8300
[115] train loss=0.5844 acc=0.9798 | val loss=0.8442 acc=0.8000
[116] train lo

In [17]:
model.load_state_dict(torch.load(f"{CKPT_DIR}/best.pt")["model"])

<All keys matched successfully>

time: 81.5 ms (started: 2026-04-21 02:53:15 +07:00)


In [18]:
import torch
import numpy as np
from collections import defaultdict
from torch.utils.data import DataLoader

# ── 1. Collect predictions ────────────────────────────────────────────────────

def collect_predictions(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            xyz    = batch["xyz"].to(device)
            norms  = batch["normals"].to(device)
            images = batch["images"].to(device)
            labels = batch["label"]

            logits = model(xyz, norms, images)
            preds  = logits.argmax(dim=-1).cpu()

            all_preds.append(preds)
            all_labels.append(labels)

    return torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy()


# ── 2. Confusion matrix ───────────────────────────────────────────────────────

def confusion_matrix(preds, labels, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    for p, l in zip(preds, labels):
        cm[l, p] += 1          # row = true, col = predicted
    return cm


# ── 3. Per-class metrics ──────────────────────────────────────────────────────

def per_class_metrics(cm):
    n = cm.shape[0]
    tp = np.diag(cm)                        # true positives per class
    fp = cm.sum(axis=0) - tp               # predicted as class i, but wrong
    fn = cm.sum(axis=1) - tp               # actual class i, but missed

    precision = np.where((tp + fp) > 0, tp / (tp + fp), 0.0)
    recall    = np.where((tp + fn) > 0, tp / (tp + fn), 0.0)
    f1        = np.where((precision + recall) > 0,
                         2 * precision * recall / (precision + recall), 0.0)

    return precision, recall, f1


# ── 4. Summary metrics ────────────────────────────────────────────────────────

def summary_metrics(cm, precision, recall, f1):
    total    = cm.sum()
    correct  = np.diag(cm).sum()

    oa          = correct / total                   # overall accuracy
    macro_prec  = precision.mean()
    macro_rec   = recall.mean()
    macro_f1    = f1.mean()

    # mean class accuracy (average of per-class recall — standard for ModelNet40)
    mca = recall.mean()

    return {
        "overall_accuracy": oa,
        "mean_class_accuracy": mca,
        "macro_precision": macro_prec,
        "macro_recall": macro_rec,
        "macro_f1": macro_f1,
    }


# ── 5. Pretty-print report ────────────────────────────────────────────────────

def print_report(class_names, precision, recall, f1, summary):
    print(f"\n{'='*60}")
    print(f"  Overall accuracy    : {summary['overall_accuracy']*100:.2f}%")
    print(f"  Mean class accuracy : {summary['mean_class_accuracy']*100:.2f}%")
    print(f"  Macro precision     : {summary['macro_precision']*100:.2f}%")
    print(f"  Macro recall        : {summary['macro_recall']*100:.2f}%")
    print(f"  Macro F1            : {summary['macro_f1']*100:.2f}%")
    print(f"{'='*60}")
    print(f"\n{'Class':<16} {'Precision':>10} {'Recall':>10} {'F1':>10}")
    print("-" * 50)
    for i, name in enumerate(class_names):
        print(f"{name:<16} {precision[i]*100:>9.1f}% {recall[i]*100:>9.1f}% {f1[i]*100:>9.1f}%")


# ── 6. Save confusion matrix as PNG ──────────────────────────────────────────

def save_confusion_matrix(cm, class_names, path="confusion_matrix.png"):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(14, 12))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right", fontsize=7)
    ax.set_yticklabels(class_names, fontsize=7)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    plt.colorbar(im, ax=ax, fraction=0.03)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"Saved confusion matrix → {path}")




time: 1.17 ms (started: 2026-04-21 02:53:16 +07:00)


In [19]:
preds, labels = collect_predictions(model, val_loader, device)

cm                    = confusion_matrix(preds, labels, n_classes=10)
precision, recall, f1 = per_class_metrics(cm)
summary               = summary_metrics(cm, precision, recall, f1)

print_report(val_ds.class_names, precision, recall, f1, summary)
print(cm)



  Overall accuracy    : 81.00%
  Mean class accuracy : 81.00%
  Macro precision     : 82.59%
  Macro recall        : 81.00%
  Macro F1            : 81.10%

Class             Precision     Recall         F1
--------------------------------------------------
bathtub               88.9%      80.0%      84.2%
bed                   83.3%     100.0%      90.9%
chair                100.0%      90.0%      94.7%
desk                  63.6%      70.0%      66.7%
dresser               64.3%      90.0%      75.0%
monitor              100.0%      80.0%      88.9%
night                 60.0%      60.0%      60.0%
sofa                  90.0%      90.0%      90.0%
table                 85.7%      60.0%      70.6%
toilet                90.0%      90.0%      90.0%
[[ 8  0  0  0  0  0  0  1  0  1]
 [ 0 10  0  0  0  0  0  0  0  0]
 [ 0  1  9  0  0  0  0  0  0  0]
 [ 0  1  0  7  0  0  2  0  0  0]
 [ 0  0  0  0  9  0  1  0  0  0]
 [ 0  0  0  0  1  8  1  0  0  0]
 [ 0  0  0  0  3  0  6  0  1  0]
 [ 1  0  0 

In [19]:
save_confusion_matrix(cm, val_ds.class_names)

Saved confusion matrix → confusion_matrix.png
time: 249 ms (started: 2026-04-21 01:47:25 +07:00)
